In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, BaseMessage
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("api_key")
model = os.getenv("model")
base_url= os.getenv("base_url")
base_url

In [ ]:
class AgentState(TypedDict):
    message : List[BaseMessage]

# llm = ChatOllama(model="gemma3:4b")
llm = ChatNVIDIA(model=model,api_key=api_key,base_url=base_url)


In [ ]:
def process (state:AgentState)->AgentState:
    response = llm.invoke(state['message'])
    print(f"AI: {response.content}")
    state['message'].append(response)
    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("process",process)
graph.add_edge(START,"process")
graph.add_edge("process",END)

agent=graph.compile()

In [ ]:
user_input = input("USER Input:")
while user_input != "exit" or "^C" :
    agent.invoke({'message':[HumanMessage(content=user_input)]})
    user_input = input("Enter:")

KeyboardInterrupt: 